In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import pyvista as pv
from mat4py import loadmat
import kagglehub

In [ ]:
path1 = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews") + '/recipes.csv'
df1 = pd.read_csv(path1)

path2 = kagglehub.dataset_download("realalexanderwei/food-com-recipes-with-ingredients-and-tags")  + '/recipes_ingredients.csv'
df2 = pd.read_csv(path2)

# Ensure matching data types
df1['RecipeId'] = pd.to_numeric(df1['RecipeId'], errors='coerce')
df2['id'] = pd.to_numeric(df2['id'], errors='coerce')

# Drop duplicates: keep only the first ingredient entry per recipe
df2_unique = df2[['id', 'ingredients_raw', 'ingredients']].drop_duplicates(subset='id')

# Merge
df_recipes = df1.merge(df2_unique, left_on='RecipeId', right_on='id', how='left')
df_recipes.drop(columns='id', inplace=True)

In [7]:
# Load the CSV files
df_burgers = pd.read_csv("isburger_llm.csv")

# Ensure isburger_llm is treated as numeric (coerce errors to NaN)
df_burgers['isburger_llm'] = pd.to_numeric(df_burgers['isburger_llm'], errors='coerce')

# Combine both dataframes side-by-side
df_combined = pd.concat([df_recipes, df_burgers], axis=1)

# Filter rows where 'isburger_llm' is not zero and not NaN
df_burger_only = df_combined[(df_combined['isburger_llm'] != 0) & (df_combined['isburger_llm'].notna())]

# Randomly sample 100 entries
sampled = df_burger_only.sample(n=100, random_state=42)

# Print the corresponding 'Name' column
for name, desc in zip(sampled['Name'], sampled['Description']):
    print(f"Name: {name}")
    print(f"Description: {desc}")
    print("-" * 40)  # Separator for readability

Name: Feta Cheese Burger
Description: Make and share this Feta Cheese Burger recipe from Food.com.
----------------------------------------
Name: Chicken Melt
Description: I enjoyed this sandwich at The Smokestack in Thurber, TX. It is a chicken version of the popular patty melt. Sort of a chicken grilled cheese. Prep time includes grilling a chicken breast.
----------------------------------------
Name: Chicken Sliders With Spicy BBQ Mayo
Description: I saw this recipe on one of Sandra Lee's shows and I had to try it.  I have made some modifications to her recipe.  My husband absolutely loves this, and I have made it for dinner using hamburger buns instead of the smaller &quot;slider&quot; rolls.  Make this for any group of people and I guarantee it will be a hit - just make sure to have plenty of napkins on hand since eating these can get a little messy!  On a side note - if you don't have time to make the chicken, a store-bought rotisserie one works fine in a pinch!
----------------

False positives: 27/100

#### Check the filtered burger recipes

In [17]:
df = pd.read_csv('filtered_burgers.csv')

sampled = df.sample(n=100, random_state=42)

# Print the corresponding 'Name' column
for name, desc in zip(sampled['Name'], sampled['Description']):
    print(f"Name: {name}")
    print("-" * 40)  # Separator for readability

Name: Dog' N Suds Grilled Pizza Burgers
----------------------------------------
Name: Hot Stuffed Mama Burger
----------------------------------------
Name: Crock Pot &quot; Cheeseburgers &quot; (Slow Cooker)
----------------------------------------
Name: Turkey Burgers
----------------------------------------
Name: Blue Bird Burgers
----------------------------------------
Name: Bacon Cheeseburgers With French Onion Dip
----------------------------------------
Name: Fresh Salmon Burger
----------------------------------------
Name: Inside-Out Bacon Cheeseburgers
----------------------------------------
Name: Barbequed Cowboy Burgers
----------------------------------------
Name: Juicy Hamburgers
----------------------------------------
Name: Carrot Burgers
----------------------------------------
Name: Biggest Loser Burger
----------------------------------------
Name: Great American Volumetric Burger
----------------------------------------
Name: Spicy Black Bean-Garlic Burgers
----

False positives: 0/100

### Check burger ingredients

In [5]:
with open('burger_ingredients.pkl', 'rb') as f:
    ingr_data = pickle.load(f)
burgers = pd.read_csv("burgers.csv")

In [18]:
i = 0
while i < 100:
    idx_recipe = np.random.randint(low=0, high=len(burgers)-1)
    ingr_list = eval(burgers.iloc[idx_recipe]['ingredients_raw'])
    idx_ingr = np.random.randint(low=0, high=len(ingr_list))

    print(ingr_list[idx_ingr])
    print(ingr_data[idx_recipe][idx_ingr])
    print('-'*150)

    i+= 1

1/2      onion, cut into wedges 
 "0.5, -, onion"
------------------------------------------------------------------------------------------------------------------------------------------------------
3 1/2  lbs    ground chuck
 "3.5, lb, chuck"
------------------------------------------------------------------------------------------------------------------------------------------------------
1/2  cup    oats
 0.5, cup, oats
------------------------------------------------------------------------------------------------------------------------------------------------------
1       egg
 1, egg, egg
------------------------------------------------------------------------------------------------------------------------------------------------------
4       rolls or 4       hamburger buns, split, toasted 
 NaN, -, rolls
------------------------------------------------------------------------------------------------------------------------------------------------------
1/2  cup   generous 

mistakes: 7/100 (serious mistakes only. Not counting easily noticed mistakes.)